<a href="https://colab.research.google.com/github/Sayak-coder/HateSpeech-detection/blob/main/Subtask_A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
df_EN = pd.read_csv('/content/StereoQueerEval_EN_training.tsv', sep='\t')
df_EN.head()

,StereoQueerEval_id,yt_title,yt_description,yt_comment,stereotype,hate_speech,target
0,training_EN_0001,[CHANNEL]: Lesbian denied communion at funeral,[CHANNEL] speaks to Barbara Johnson who says a...,She's living in unrepentant defiance to God's ...,yes,yes_explicit,individual_l
1,training_EN_0002,GOP Congresswoman uses anti-trans slur in Hous...,Rep. Nancy Mace (R-SC) was challenged by Rep. ...,Rep Mace: Keep on being dedicated to truth and...,yes,yes_implicit,group_t
2,training_EN_0003,Kevin Smith defends his gay brother,"Kevin Smith supports gay rights, and defends h...","We all need Jesus, He said no homosexual shall...",yes,yes_implicit,"group_l,g"
3,training_EN_0004,Judge removes child from gay foster parents,A Utah judge removed a foster child from a les...,So can two black men adopt a little white girl...,yes,yes_implicit,group_l
4,training_EN_0005,Chechen leader Ramzan Kadyrov questioned on 'g...,Chechnya's leader Ramzan Kadyrov is questioned...,Of course west and US will hate him,yes,yes_implicit,group_g


# Experiment 1
only comment as a input

In [3]:
df = df_EN[['yt_comment','stereotype']]
df.head()

,yt_comment,stereotype
0,She's living in unrepentant defiance to God's ...,yes
1,Rep Mace: Keep on being dedicated to truth and...,yes
2,"We all need Jesus, He said no homosexual shall...",yes
3,So can two black men adopt a little white girl...,yes
4,Of course west and US will hate him,yes


In [4]:
X = df['yt_comment']
y = df['stereotype']

In [5]:
y = df['stereotype'].map({'no': 0, 'yes': 1}).tolist()

In [6]:
X_train,X_test,y_train,y_test = train_test_split(X,y, test_size=0.2, random_state=45)

Transformer

In [ ]:
!pip install transformers

### DistilBert

In [ ]:
from transformers import DistilBertTokenizerFast
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased-finetuned-sst-2-english')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [ ]:
train_encodings = tokenizer(X_train, truncation= True, padding=True, max_length=128)
test_encodings = tokenizer(X_test, truncation= True, padding=True, max_length=128)

In [ ]:
train_encodings

{'input_ids': [[101, 2307, 3105, 3163, 999, 2017, 2031, 4217, 2293, 2058, 5223, 999, 1996, 3412, 2157, 1005, 1055, 2502, 4140, 2854, 2038, 2579, 1037, 2718, 1012, 2023, 5621, 2003, 1037, 13301, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [101, 2064, 2016, 9790, 2005, 27652, 1029, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [101, 2515, 3087, 3305, 2054, 1996, 4259, 2457, 2106, 1029, 2027, 2165, 1996, 3546, 2942, 2916, 2

In [ ]:
import torch

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TextDataset(train_encodings, y_train)
test_dataset = TextDataset(test_encodings, y_test)

In [ ]:
train_dataset

In [ ]:
from transformers import (
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=60,
    weight_decay=0.01,
    logging_steps=10,
)

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased-finetuned-sst-2-english", num_labels=2
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,3.365893
20,2.108164
30,2.191133
40,1.426385
50,0.761148
60,0.633770
70,0.761159
80,0.717413
90,0.664416
100,0.696565


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=598, training_loss=0.6434753120543566, metrics={'train_runtime': 3326.3509, 'train_samples_per_second': 1.438, 'train_steps_per_second': 0.18, 'total_flos': 158364775093248.0, 'train_loss': 0.6434753120543566, 'epoch': 2.0})

In [ ]:
trainer.evaluate(test_dataset)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step
0.345524,0.674712,598


{'eval_loss': 0.6747122406959534}

In [ ]:
y_pred = np.argmax(trainer.predict(test_dataset).predictions, axis=-1)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
output = trainer.predict(test_dataset)[1]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print(cm)

[[253  76]
 [ 98 171]]


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.72      0.77      0.74       329
           1       0.69      0.64      0.66       269

    accuracy                           0.71       598
   macro avg       0.71      0.70      0.70       598
weighted avg       0.71      0.71      0.71       598



###cardiffnlp/twitter-roberta-base-hate-latest

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=45)
X_val,X_test,y_val,y_test = train_test_split(X_test,y_test,test_size=0.5,random_state=45)

In [ ]:
def build_inputs(d):
    comments = d['yt_comment'].tolist()

def encode(d, tokenizer, max_length=128):
    comments = build_inputs(d)
    return tokenizer(comments, truncation=True, max_length=max_length)

In [ ]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
from transformers import AutoTokenizer

MODEL = f"cardiffnlp/twitter-roberta-base-hate-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

config.json:   0%|          | 0.00/888 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

In [ ]:
# Since X_train, X_val, X_test are Series containing the comments directly, we can convert them to lists
train_encodings = tokenizer(X_train.tolist(), truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(X_val.tolist(), truncation=True, padding=True, max_length=128)
test_encodings  = tokenizer(X_test.tolist(),  truncation=True, padding=True, max_length=128)

# y_train, y_val, y_test are already lists, so we don't need to call .tolist()
train_dataset = TextDataset(train_encodings, y_train)
val_dataset = TextDataset(val_encodings, y_val)
test_dataset  = TextDataset(test_encodings,  y_test)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=60,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    max_grad_norm=1.0
)

In [ ]:
from transformers import AutoModelForSequenceClassification

MODEL = "cardiffnlp/twitter-roberta-base-hate-latest"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=2,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "yes"},
    label2id={"no": 0, "yes": 1},
).to(device)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  499MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_twitter_roberta = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_twitter_roberta.train()

Step,Training Loss
10,1.786205
20,1.248520
30,0.985178
40,0.695024
50,0.621915
60,0.664766
70,0.563414
80,0.616044
90,0.573347
100,0.592933


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=600, training_loss=0.4188158592581749, metrics={'train_runtime': 301.7221, 'train_samples_per_second': 31.698, 'train_steps_per_second': 1.989, 'total_flos': 629098533365760.0, 'train_loss': 0.4188158592581749, 'epoch': 4.0})

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_twitter_roberta.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","yes",], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.761     0.775     0.768       173
         yes      0.683     0.667     0.675       126

    accuracy                          0.729       299
   macro avg      0.722     0.721     0.721       299
weighted avg      0.728     0.729     0.729       299

[[134  39]
 [ 42  84]]


#Experiment 2
comment + title as input

In [ ]:
df = df_EN

In [ ]:
X = (df.drop(['StereoQueerEval_id', 'yt_description', 'stereotype', 'target', 'hate_speech'], axis=1))
y = df['stereotype'].map({'no':0, 'yes':1})

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=45)
X_val,X_test,y_val,y_test = train_test_split(X_test,y_test,test_size=0.5,random_state=45)

In [ ]:
def build_inputs(d):
    comments = d['yt_comment'].tolist()
    title = d['yt_title'].fillna("")
    ctx = title
    return comments, ctx.tolist()

def encode(d, tokenizer, max_length=256):
    comments, ctx = build_inputs(d)
    return tokenizer(comments, ctx, truncation="longest_first", max_length=max_length)

In [ ]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

###distilbert

In [ ]:
from transformers import DistilBertTokenizerFast
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased-finetuned-sst-2-english')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [ ]:
train_encodings = encode(X_train, tokenizer)
val_encodings = encode(X_val, tokenizer)
test_encodings  = encode(X_test,  tokenizer)

train_dataset = TextDataset(train_encodings, y_train.tolist())
val_dataset = TextDataset(val_encodings, y_val.tolist())
test_dataset  = TextDataset(test_encodings,  y_test.tolist())

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=60,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    max_grad_norm=1.0
)

In [ ]:
from transformers import DistilBertForSequenceClassification

MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"

# Correctly load the pre-trained model and configure it in one step
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "yes"},
    label2id={"no": 0, "yes": 1}
).to(device)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [ ]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_distilbert = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_distilbert.train()

Step,Training Loss
10,3.030755
20,2.035303
30,2.142467
40,1.619827
50,1.296764
60,0.788748
70,0.848426
80,0.745982
90,0.654785
100,0.666081


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1196, training_loss=0.4976763746212159, metrics={'train_runtime': 164.5145, 'train_samples_per_second': 58.135, 'train_steps_per_second': 7.27, 'total_flos': 310997489215356.0, 'train_loss': 0.4976763746212159, 'epoch': 4.0})

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_distilbert.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","yes"], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.724     0.728     0.726       173
         yes      0.624     0.619     0.622       126

    accuracy                          0.682       299
   macro avg      0.674     0.674     0.674       299
weighted avg      0.682     0.682     0.682       299

[[126  47]
 [ 48  78]]


###cardiffnlp/twitter-roberta-base-hate-latest

In [ ]:
from transformers import AutoTokenizer

MODEL = f"cardiffnlp/twitter-roberta-base-hate-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

In [ ]:
train_encodings = encode(X_train, tokenizer)
val_encodings = encode(X_val, tokenizer)
test_encodings  = encode(X_test,  tokenizer)

train_dataset = TextDataset(train_encodings, y_train.tolist())
val_dataset = TextDataset(val_encodings, y_val.tolist())
test_dataset  = TextDataset(test_encodings,  y_test.tolist())

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=60,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    max_grad_norm=1.0
)

In [ ]:
from transformers import AutoModelForSequenceClassification

MODEL = "cardiffnlp/twitter-roberta-base-hate-latest"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=2,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "yes"},
    label2id={"no": 0, "yes": 1},
).to(device)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_twitter_roberta = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_twitter_roberta.train()

Step,Training Loss
10,1.831212
20,1.140121
30,0.982738
40,0.603205
50,0.606738
60,0.641380
70,0.590756
80,0.604328
90,0.581529
100,0.599527


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step,Training Loss
10,1.831212
20,1.140121
30,0.982738
40,0.603205
50,0.606738
60,0.641380
70,0.590756
80,0.604328
90,0.581529
100,0.599527


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=600, training_loss=0.42130106170972187, metrics={'train_runtime': 302.883, 'train_samples_per_second': 31.577, 'train_steps_per_second': 1.981, 'total_flos': 790760749110840.0, 'train_loss': 0.42130106170972187, 'epoch': 4.0})

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_twitter_roberta.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","yes",], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.750     0.780     0.765       173
         yes      0.681     0.643     0.661       126

    accuracy                          0.722       299
   macro avg      0.715     0.712     0.713       299
weighted avg      0.721     0.722     0.721       299

[[135  38]
 [ 45  81]]


#Experiment 3
title + description + comment as input

In [13]:
import torch

In [14]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [8]:
df = df+df_EN

In [21]:
X = (df.drop(['StereoQueerEval_id', 'stereotype', 'target', 'hate_speech'], axis=1))
y = df['stereotype'].map({'no':0, 'yes':1})

In [22]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=45)
X_val,X_test,y_val,y_test = train_test_split(X_test,y_test,test_size=0.5,random_state=45)

In [23]:
def build_inputs(d):
    comments = d['yt_comment'].tolist()
    title = d['yt_title'].fillna("")
    desc  = d['yt_description'].fillna("").str[:300]        # trim long descriptions
    ctx = (title + " . " + desc)
    return comments, ctx.tolist()

def encode(d, tokenizer, max_length=256):
    comments, ctx = build_inputs(d)
    return tokenizer(comments, ctx, truncation="longest_first", max_length=max_length)

In [24]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

###distilbert

In [16]:
from transformers import DistilBertTokenizerFast
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased-finetuned-sst-2-english')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [25]:
train_encodings = encode(X_train, tokenizer)
val_encodings = encode(X_val, tokenizer)
test_encodings  = encode(X_test,  tokenizer)

train_dataset = TextDataset(train_encodings, y_train.tolist())
val_dataset = TextDataset(val_encodings, y_val.tolist())
test_dataset  = TextDataset(test_encodings,  y_test.tolist())

In [26]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=60,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    max_grad_norm=1.0
)

In [28]:
from transformers import DistilBertForSequenceClassification

MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"

# Correctly load the pre-trained model and configure it in one step
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "yes"},
    label2id={"no": 0, "yes": 1}
).to(device)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [29]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_distilbert = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_distilbert.train()

Step,Training Loss
10,2.580326
20,1.860109
30,1.757615
40,1.253429
50,1.149276
60,0.773118
70,0.812557
80,0.739083
90,0.668637
100,0.692311


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1196, training_loss=0.4972764420070776, metrics={'train_runtime': 190.481, 'train_samples_per_second': 50.21, 'train_steps_per_second': 6.279, 'total_flos': 438680537996460.0, 'train_loss': 0.4972764420070776, 'epoch': 4.0})

In [30]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_distilbert.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","yes",], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.716     0.728     0.722       173
         yes      0.618     0.603     0.610       126

    accuracy                          0.676       299
   macro avg      0.667     0.666     0.666       299
weighted avg      0.675     0.676     0.675       299

[[126  47]
 [ 50  76]]


In [31]:
from transformers import AutoTokenizer

MODEL = f"cardiffnlp/twitter-roberta-base-hate-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

In [32]:
train_encodings = encode(X_train, tokenizer)
val_encodings = encode(X_val, tokenizer)
test_encodings  = encode(X_test,  tokenizer)

train_dataset = TextDataset(train_encodings, y_train.tolist())
val_dataset = TextDataset(val_encodings, y_val.tolist())
test_dataset  = TextDataset(test_encodings,  y_test.tolist())

In [33]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=60,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    max_grad_norm=1.0
)

In [34]:
from transformers import AutoModelForSequenceClassification

MODEL = "cardiffnlp/twitter-roberta-base-hate-latest"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=2,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "yes"},
    label2id={"no": 0, "yes": 1},
).to(device)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  499MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [35]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_twitter_roberta = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_twitter_roberta.train()

Step,Training Loss
10,2.126940
20,1.124178
30,0.971714
40,0.667201
50,0.633121
60,0.675282
70,0.587275
80,0.630947
90,0.594619
100,0.624556


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=600, training_loss=0.4296068575978279, metrics={'train_runtime': 430.8706, 'train_samples_per_second': 22.197, 'train_steps_per_second': 1.393, 'total_flos': 1026698393562000.0, 'train_loss': 0.4296068575978279, 'epoch': 4.0})

In [36]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_twitter_roberta.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","yes",], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.757     0.757     0.757       173
         yes      0.667     0.667     0.667       126

    accuracy                          0.719       299
   macro avg      0.712     0.712     0.712       299
weighted avg      0.719     0.719     0.719       299

[[131  42]
 [ 42  84]]
